# Compose Dataset (high-fidelity contours)

This notebook composites instruments onto backgrounds and generates YOLO polygon labels with improved contour fidelity (supersampling + blur + light erosion + adaptive simplify).


In [45]:
# Install (run once)
import sys
!{sys.executable} -m pip install -q opencv-python numpy pillow



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: /usr/local/Cellar/jupyterlab/4.3.1_1/libexec/bin/python -m pip install --upgrade pip


In [46]:
# Configuration
from pathlib import Path

BASE_DIR = Path("/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator").resolve()
FG_DIR = BASE_DIR / "photos_wo_background"
BG_DIR = BASE_DIR / "backgrounds"
OUT_IMAGES = BASE_DIR / "dataset_out" / "images"
OUT_SEG0 = BASE_DIR / "dataset_out" / "labels_seg0"
OUT_CLS30 = BASE_DIR / "dataset_out" / "labels_cls30"
CLASS_MAP_PATH = BASE_DIR / "dataset_out" / "class_mapping.json"

# Canonical contour extraction params
GAUSS_SIGMA = 0.8  # slight blur on original alpha
ERODE_ITER = 1     # light erosion to remove halo
MAX_VERTICES = 256 # polygon cap after simplify
SIMPLIFY_ERR_PX = 0.75

# Dataset balancing
MIN_PER_TOOL = 5
PER_IMAGE_MIN = 1
PER_IMAGE_MAX = 1
MAX_PLACE_TRIES = 30

BASE_DIR, FG_DIR, BG_DIR, OUT_IMAGES


(PosixPath('/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator'),
 PosixPath('/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/photos_wo_background'),
 PosixPath('/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/backgrounds'),
 PosixPath('/Users/wladyslaw/Documents/Job/LCT/additional_aero_tools_generator/dataset_out/images'))

In [47]:
# Helpers
import json
import random
import uuid
from typing import Dict, List, Tuple

import cv2
import numpy as np
from PIL import Image, ImageOps, ImageEnhance

SUPPORTED_BG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
SUPPORTED_FG_EXTS = {".png"}

# Rotation params
ROTATE_MIN_DEG = -45
ROTATE_MAX_DEG = 45
ROTATE_DIAGONAL_THRESHOLD = 30  # deg, apply diagonal factor when |angle| >= this
DIAGONAL_FACTOR = 0.7

# Size rules (max length in px) per tool code
TOOL_SIZE_RULES: Dict[str, Tuple[int, int]] = {
    # 1. ОТВЕРТКИ И ДЕРЖАТЕЛИ (450-800)
    "t45": (700, 800), "4622": (700, 800), "4632": (700, 800),
    "13001": (650, 750), "13009": (650, 750),
    "4000": (600, 700), "4008": (550, 650), "412": (450, 550), "443": (450, 550),
    # 2. НАБОРЫ И КОМПЛЕКТЫ (300)
    "107679": (300, 300),
    # 3. ТОРЦЕВЫЕ ГОЛОВКИ И НАСАДКИ (50-140)
    "45": (50, 100), "46": (50, 100), "40": (70, 80), "40L": (100, 120), "402": (120, 140),
    # 4. РУКОЯТКИ ТОРЦЕВЫЕ (350-450)
    "400n": (350, 450),
    # 5. ПРОБОЙНИКИ (300-500)
    "104": (300, 400), "109": (300, 400), "12495": (350, 500),
    # 6. УДЛИНИТЕЛИ (по фактической длине)
    "405qr2": (180, 250), "427qr3": (250, 350), "405qr6": (400, 550), "427qr10": (600, 750),
    # 7. КАРДАНЫ (250-400)
    "407qr": (250, 350), "428qr": (300, 400),
    # 8. ПЕРЕХОДНИКИ (100-150)
    "409m": (100, 120), "431": (100, 120), "432m": (120, 150),
    # 9. ТРЕЩОТКИ (400-600)
    "415qr": (400, 550), "435qr": (450, 600), "415sgb": (400, 550),
    # 10. ДИНАМОМЕТРИЧЕСКИЕ ИНСТРУМЕНТЫ (750-900)
    "10678": (800, 900), "453": (750, 850),
    # 11. РАЗВОДНЫЕ И РЕГУЛИРУЕМЫЕ (600-750)
    "4025": (600, 750),
    # 12. ИЗМЕРИТЕЛЬНЫЕ ИНСТРУМЕНТЫ (600-800)
    "12900": (700, 800), "129003": (600, 700),
    # 13. РЕЖУЩИЕ ИНСТРУМЕНТЫ (550-650)
    "65515240": (550, 650), "65516240": (550, 650), "6707": (600, 650),
    # 14. ГАЕЧНЫЕ КЛЮЧИ (500-700)
    "65642175": (600, 700), "13": (500, 600), "16": (600, 700), "25": (500, 600),
    # 15. ПАССАТИЖИ И ПЛОСКОГУБЦЫ (600-650)
    "65751220": (600, 650), "6700": (600, 650), "6731": (600, 650), "65165200": (600, 650), "65325170": (600, 650), "66005160": (600, 650),
    # 16. МОЛОТКИ (500-650)
    "10959": (500, 600), "10961": (550, 650),
    # 17. СПЕЦИАЛЬНЫЕ ИНСТРУМЕНТЫ (300-700)
    "11095": (300, 400), "12321": (400, 500), "12921m": (600, 700), "12922": (500, 600), "13110": (700, 800), "13128": (350, 450),
}

SMALL_DETAILS = {"45", "46", "40", "40L", "402", "409m", "431", "432m"}
ADAPTERS = {"409m", "431", "432m"}


def ensure_dirs():
    OUT_IMAGES.mkdir(parents=True, exist_ok=True)
    OUT_SEG0.mkdir(parents=True, exist_ok=True)
    OUT_CLS30.mkdir(parents=True, exist_ok=True)


def list_images(directory: Path, exts: set) -> List[Path]:
    return [p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in exts]


def load_background() -> Image.Image:
    images = list_images(BG_DIR, SUPPORTED_BG_EXTS)
    if not images:
        raise RuntimeError(f"No background images found in: {BG_DIR}")
    bg_path = random.choice(images)
    bg = ImageOps.exif_transpose(Image.open(bg_path)).convert("RGB")
    return bg


def load_foregrounds() -> List[Path]:
    images = list_images(FG_DIR, SUPPORTED_FG_EXTS)
    if not images:
        raise RuntimeError(f"No transparent PNGs found in: {FG_DIR}")
    return images

# cache for canonical contours at original scale
_CANONICAL_CACHE: Dict[str, np.ndarray] = {}


def gaussian_blur_uint8(img: np.ndarray, sigma: float) -> np.ndarray:
    if sigma <= 0:
        return img
    k = int(max(3, 2 * int(3 * sigma) + 1))
    return cv2.GaussianBlur(img, (k, k), sigmaX=sigma, sigmaY=sigma, borderType=cv2.BORDER_REPLICATE)


def simplify_poly_points(points: np.ndarray, max_vertices: int, max_err_px: float) -> np.ndarray:
    if len(points) <= max_vertices:
        return points
    eps_low, eps_high = 0.0, max_err_px
    best = points
    for _ in range(12):
        eps = (eps_low + eps_high) / 2.0
        approx = cv2.approxPolyDP(points.reshape(-1, 1, 2), eps, True).reshape(-1, 2)
        if len(approx) > max_vertices:
            eps_low = eps
        else:
            best = approx
            eps_high = eps
    return best


def compute_canonical_contour(fg_rgba: Image.Image) -> np.ndarray:
    # alpha from original PNG
    alpha = np.array(fg_rgba.split()[-1])
    # optional: slight blur and erosion to remove halos from background removal
    if GAUSS_SIGMA > 0:
        alpha = gaussian_blur_uint8(alpha, GAUSS_SIGMA)
    _, mask = cv2.threshold(alpha, 0, 255, cv2.THRESH_BINARY)
    if ERODE_ITER > 0:
        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.erode(mask, kernel, iterations=ERODE_ITER)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.empty((0, 2), dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt[:, 0, :].astype(np.float32)
    pts = simplify_poly_points(pts, MAX_VERTICES, SIMPLIFY_ERR_PX)
    return pts


def compute_contour_from_alpha(alpha_u8: np.ndarray) -> np.ndarray:
    # Extract contour from an arbitrary alpha mask (e.g., after rotation)
    a = alpha_u8
    if GAUSS_SIGMA > 0:
        a = gaussian_blur_uint8(a, GAUSS_SIGMA)
    _, mask = cv2.threshold(a, 0, 255, cv2.THRESH_BINARY)
    if ERODE_ITER > 0:
        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.erode(mask, kernel, iterations=ERODE_ITER)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.empty((0, 2), dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt[:, 0, :].astype(np.float32)
    pts = simplify_poly_points(pts, MAX_VERTICES, SIMPLIFY_ERR_PX)
    return pts


def compute_target_long_side(tool_code: str, role: str = "main", angle_deg: float = 0.0) -> int:
    # role: main=1.0, secondary=0.75, small=0.6 by spec
    unknown = tool_code not in TOOL_SIZE_RULES
    min_px, max_px = TOOL_SIZE_RULES.get(tool_code, (100, 140))
    size = random.randint(min_px, max_px)
    factor = 1.0 if role == "main" else (0.75 if role == "secondary" else 0.6)
    size = int(size * factor)
    # diagonal factor when rotated significantly
    if abs(angle_deg) >= ROTATE_DIAGONAL_THRESHOLD:
        size = int(size * DIAGONAL_FACTOR)
    # if tool not in rules, make it 10x larger
    if unknown:
        size *= 10
    return max(1, size)


def resize_by_long_side(img: Image.Image, target_long: int) -> Image.Image:
    w, h = img.size
    long_side = max(w, h)
    if long_side == 0:
        return img
    scale = target_long / long_side
    target_w = max(1, int(round(w * scale)))
    target_h = max(1, int(round(h * scale)))
    # Enforce minimal thickness 30px
    if min(target_w, target_h) < 30:
        pad = 30 / min(w, h)
        target_w = max(target_w, int(round(w * pad)))
        target_h = max(target_h, int(round(h * pad)))
    return img.resize((target_w, target_h), resample=Image.LANCZOS)


def rotate_image_with_alpha(img: Image.Image, angle_deg: float) -> Image.Image:
    if angle_deg == 0:
        return img
    return img.rotate(angle_deg, resample=Image.BICUBIC, expand=True)


def random_photometric(img_rgba: Image.Image) -> Image.Image:
    # Split channels
    r, g, b, a = img_rgba.split()
    rgb = Image.merge("RGB", (r, g, b))

    # Stronger ranges (≈2–3x)
    # Brightness (0.6–1.4)
    rgb = ImageEnhance.Brightness(rgb).enhance(random.uniform(0.6, 1.4))
    # Contrast (0.6–1.6)
    rgb = ImageEnhance.Contrast(rgb).enhance(random.uniform(0.6, 1.6))
    # Saturation (0.6–1.6)
    rgb = ImageEnhance.Color(rgb).enhance(random.uniform(0.6, 1.6))

    # Hue shift (-15..+15 degrees) via HSV
    hsv = np.array(rgb.convert("HSV"), dtype=np.uint8)
    hue_shift = int(random.uniform(-15, 15)) % 180
    hsv[:, :, 0] = (hsv[:, :, 0].astype(int) + hue_shift) % 180
    rgb = Image.fromarray(hsv, mode="HSV").convert("RGB")

    # Gamma (0.75–1.25)
    gamma = random.uniform(0.75, 1.25)
    inv = 1.0 / max(1e-6, gamma)
    lut = np.array([(i / 255.0) ** inv * 255 for i in range(256)]).clip(0, 255).astype(np.uint8)
    rgb = Image.fromarray(cv2.LUT(np.array(rgb), lut))

    # Color temperature shift (±20%) using RB gains
    temp = random.uniform(-0.20, 0.20)  # -cool..+warm
    arr = np.array(rgb).astype(np.float32)
    if temp >= 0:
        arr[:, :, 0] *= (1 + temp)        # R boost
        arr[:, :, 2] *= (1 - temp * 0.5)  # B reduce
    else:
        arr[:, :, 2] *= (1 - temp)        # B boost (temp negative)
        arr[:, :, 0] *= (1 + temp * 0.5)  # R reduce

    # Strong shadow/vignette with probability ~0.6
    if random.random() < 0.6:
        h, w = arr.shape[:2]
        shadow = np.zeros((h, w), dtype=np.float32)
        num = random.randint(1, 2)
        for _ in range(num):
            cx = random.randint(int(-0.1 * w), int(1.1 * w))
            cy = random.randint(int(-0.1 * h), int(1.1 * h))
            ax = random.randint(int(0.25 * w), int(0.7 * w))
            ay = random.randint(int(0.15 * h), int(0.5 * h))
            ang = random.uniform(0, 180)
            cv2.ellipse(shadow, (cx, cy), (ax, ay), ang, 0, 360, 1.0, -1)
        k = int(max(5, (min(h, w) * 0.15) // 2 * 2 + 1))
        shadow = cv2.GaussianBlur(shadow, (k, k), 0)
        strength = random.uniform(0.35, 0.75)  # final brightness inside shadow
        arr = arr * (1 - shadow[..., None] * (1 - strength))

    arr = np.clip(arr, 0, 255).astype(np.uint8)
    rgb = Image.fromarray(arr)

    # Gaussian noise stronger (sigma 0–10)
    sigma = random.uniform(0.0, 10.0)
    if sigma > 1.0:
        noisy = arr.astype(np.float32) + np.random.normal(0, sigma, arr.shape)
        rgb = Image.fromarray(np.clip(noisy, 0, 255).astype(np.uint8))

    # Merge alpha back
    return Image.merge("RGBA", (*rgb.split(), a))


def place_with_margins(bg: Image.Image, fg: Image.Image, margin: int = 60) -> Tuple[Image.Image, Tuple[int, int]]:
    bg_w, bg_h = bg.size
    w, h = fg.size
    max_x = max(margin, bg_w - w - margin)
    max_y = max(margin, bg_h - h - margin)
    x = random.randint(margin, max_x) if bg_w - 2 * margin > 0 else 0
    y = random.randint(margin, max_y) if bg_h - 2 * margin > 0 else 0
    return fg, (x, y)


def offset_and_normalize(points: np.ndarray, offset_xy, bg_size) -> List[float]:
    if points.size == 0:
        return []
    bg_w, bg_h = bg_size
    tx, ty = offset_xy
    pts = points.copy()
    pts[:, 0] = (pts[:, 0] + tx) / max(1, bg_w)
    pts[:, 1] = (pts[:, 1] + ty) / max(1, bg_h)
    flat = []
    for x, y in pts:
        flat.extend([float(min(1.0, max(0.0, x))), float(min(1.0, max(0.0, y)))])
    return flat


def polygon_to_mask(pts: np.ndarray, bg_size: Tuple[int, int]) -> np.ndarray:
    # pts in absolute pixels (x,y)
    w, h = bg_size
    mask = np.zeros((h, w), dtype=np.uint8)
    if pts.size == 0:
        return mask
    poly = np.round(pts).astype(np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [poly], 255)
    return mask


def occludes_too_much(new_abs_pts: np.ndarray, prev_abs_list: List[np.ndarray], bg_size: Tuple[int, int], threshold: float = 0.6) -> bool:
    if not prev_abs_list or new_abs_pts.size == 0:
        return False
    w, h = bg_size
    new_mask = polygon_to_mask(new_abs_pts, bg_size)
    for prev_pts in prev_abs_list:
        if prev_pts.size == 0:
            continue
        prev_mask = polygon_to_mask(prev_pts, bg_size)
        prev_area = int(prev_mask.sum() // 255)
        if prev_area <= 0:
            continue
        inter = np.logical_and(prev_mask > 0, new_mask > 0).sum()
        if inter / prev_area >= threshold:
            return True
    return False


def compose_one(bg_img: Image.Image, codes: List[str], apply_aug: bool = True, geo: List[Dict] | None = None, return_geo: bool = False):
    composed = bg_img.copy()
    bg_w, bg_h = composed.size
    placed = []
    abs_polys: List[np.ndarray] = []  # absolute polygons for occlusion
    out_geo: List[Dict] = []

    if not codes:
        return (composed, placed, out_geo) if return_geo else (composed, placed)

    def process_one(code: str, role: str, params: Dict | None, max_tries: int = MAX_PLACE_TRIES):
        nonlocal composed, placed, abs_polys, out_geo
        path = (FG_DIR / f"{code}.png")
        if not path.exists():
            return
        fg = Image.open(path).convert("RGBA")
        if apply_aug:
            fg = random_photometric(fg)
        angle = params["angle"] if params else random.uniform(ROTATE_MIN_DEG, ROTATE_MAX_DEG)
        target_long = params["target_long"] if params else compute_target_long_side(code, role=role, angle_deg=angle)
        fg_resized = resize_by_long_side(fg, target_long)
        fg_resized = rotate_image_with_alpha(fg_resized, angle)

        # After rotation/resizing, compute contour directly from alpha to avoid any drift
        alpha_now = np.array(fg_resized.split()[-1])
        rel_pts = compute_contour_from_alpha(alpha_now)

        tries = [(params["x"], params["y"]) ] if params else [None] * max_tries
        for t in tries:
            if t is None:
                fg_try, (x, y) = place_with_margins(composed, fg_resized, margin=60)
            else:
                fg_try, (x, y) = fg_resized, t
            if rel_pts.size:
                abs_pts = rel_pts.copy()
                abs_pts[:, 0] += x
                abs_pts[:, 1] += y
            else:
                abs_pts = rel_pts
            if params or not occludes_too_much(abs_pts, abs_polys, (bg_w, bg_h), threshold=0.6):
                composed.paste(fg_try, (x, y), mask=fg_try.split()[-1])
                abs_polys.append(abs_pts)
                poly_flat = offset_and_normalize(rel_pts, (x, y), (bg_w, bg_h))
                if len(poly_flat) >= 6:
                    placed.append((code, poly_flat))
                if return_geo and not params:
                    out_geo.append({"code": code, "angle": angle, "target_long": target_long, "x": x, "y": y})
                break

    if geo:
        # render by given geometry (codes order follows geo)
        for g in geo:
            code = g["code"]
            role = "small" if code in SMALL_DETAILS or code in ADAPTERS else "secondary"
            if code == codes[0]:
                role = "main"
            process_one(code, role, params=g)
    else:
        # build geometry
        if codes:
            process_one(codes[0], role="main", params=None)
        for c in codes[1:]:
            role = "small" if c in SMALL_DETAILS or c in ADAPTERS else "secondary"
            process_one(c, role=role, params=None)

    if return_geo:
        return composed, placed, out_geo
    return composed, placed


def write_yolo_seg_labels(labels_path: Path, objects: List[Tuple[int, List[float]]]):
    lines = []
    for obj in objects:
        if len(obj) == 3:
            cls_id, poly, name = obj
        else:
            cls_id, poly = obj
            name = None
        parts = [str(cls_id)] + [f"{v:.6f}" for v in poly]
        line = " ".join(parts)
        if name:
            line += f" # {name}"
        lines.append(line)
    labels_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


In [48]:
# Compose loop with balancing
ensure_dirs()

# class labels map
class_name_to_id = {}
next_id = 30

# collect tool codes from available PNGs
tool_codes = sorted([p.stem for p in FG_DIR.glob("*.png")])
counts = {c: 0 for c in tool_codes}

images_generated = 0
while any(v < MIN_PER_TOOL for v in counts.values()):
    bg_img = load_background()
    # decide how many items in this image
    n_items = random.randint(PER_IMAGE_MIN, PER_IMAGE_MAX)

    # weighted sampling towards underrepresented tools
    deficits = np.array([max(0, MIN_PER_TOOL - counts[c]) for c in tool_codes], dtype=np.float32)
    if deficits.sum() == 0:
        weights = np.ones(len(tool_codes), dtype=np.float32)
    else:
        weights = deficits / deficits.sum()

    # choose main code
    main_code = random.choices(tool_codes, weights=weights, k=1)[0]
    chosen_codes = [main_code]

    # choose others, allowing duplicates up to 3 times per image
    while len(chosen_codes) < n_items:
        c = random.choices(tool_codes, weights=weights, k=1)[0]
        if chosen_codes.count(c) < 3:
            chosen_codes.append(c)
        else:
            continue

    # first pass: build geometry without photometric augmentations
    base_composed, base_placed, geo = compose_one(bg_img, chosen_codes, apply_aug=False, return_geo=True)
    if not base_placed:
        continue

    # update counts from base pass
    for code, _ in base_placed:
        counts[code] += 1

    # labels for base pass
    objs_seg0 = []
    objs_cls30 = []
    for code, poly in base_placed:
        objs_seg0.append((0, poly, code))
        if code not in class_name_to_id:
            class_name_to_id[code] = next_id
            next_id += 1
        objs_cls30.append((class_name_to_id[code], poly, code))

    # save base image and labels
    img_id = uuid.uuid4().hex
    img_path = OUT_IMAGES / f"{img_id}.jpg"
    base_composed.save(img_path, format="JPEG", quality=95)
    write_yolo_seg_labels(OUT_SEG0 / f"{img_id}.txt", objs_seg0)
    write_yolo_seg_labels(OUT_CLS30 / f"{img_id}.txt", objs_cls30)

    # second pass: re-render with photometric augmentations using same geometry
    aug_composed, aug_placed = compose_one(bg_img, chosen_codes, apply_aug=True, geo=geo)

    # save augmented image and reuse same labels (geometry identical)
    img_id2 = uuid.uuid4().hex
    img_path2 = OUT_IMAGES / f"{img_id2}.jpg"
    aug_composed.save(img_path2, format="JPEG", quality=95)
    write_yolo_seg_labels(OUT_SEG0 / f"{img_id2}.txt", objs_seg0)
    write_yolo_seg_labels(OUT_CLS30 / f"{img_id2}.txt", objs_cls30)

    images_generated += 1
    if images_generated % 10 == 0:
        need_left = sum(max(0, MIN_PER_TOOL - counts[c]) for c in tool_codes)
        print(f"Images: {images_generated} | Remaining instances to reach 100 each: {need_left}")

# mapping
CLASS_MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
CLASS_MAP_PATH.write_text(json.dumps({str(v): k for k, v in class_name_to_id.items()}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved class mapping to: {CLASS_MAP_PATH}")


Images: 10 | Remaining instances to reach 100 each: 35
Images: 20 | Remaining instances to reach 100 each: 25
Images: 30 | Remaining instances to reach 100 each: 15


KeyboardInterrupt: 